# Experiment 0: second-campaign checks

Debugging evidence only: every pilot audit, the exact-resume, FP32-capacity and evaluation checks, monitor timing and the LGD density audit. Nothing here is a scientific result.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, campaign2 as c2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('experiment0/01_campaign2_checks')
report = cp.NotebookReport('Experiment 0: second-campaign checks', sink=sink)

## 1. Pilot audits and GPU checks

Pass/fail per pilot and per check task.

In [2]:
summary = c2.experiment0_summary()
tables = c2.experiment0_tables(summary)
print('Experiment 0 summary available:', summary is not None)
for name, frame in tables.items():
    report.table(name, frame)
report.add('1. Pilot audits and GPU checks', 'The local output has no Experiment 0 summary receipt.' if summary is None else
           'Overall gate in the available receipt: ' + str(summary.get('passed')))

Experiment 0 summary available: False


## 2. LGD density roughness

Median adjacent-bucket log ratio of each original TabPFN regressor per LGD table.

In [3]:
import pandas as pd
density = (summary or {}).get('density_audit') or {}
report.table('Density roughness', pd.DataFrame(density.get('rows', [])))
report.add('2. LGD density roughness', 'Flagging threshold for the median adjacent-bucket log ratio: 2.5.' if density else
           'The local summary contains no density measurements.')

""


## 3. Training precision against FP32

Does the gradient at update zero agree with a full-precision reference? The figure shows the measured directions of mixed-precision updates. Earlier TabPFN v2 PD receipts cover ordinary FP16 and FP16 with FP32 transformer normalizations. Each table identifies the policy in force when the probe ran.

TabPFN v2 PD now uses strict FP32 with a memory-bounded backward pass. The independent check compares gradients from native and bounded FP32 on the same smaller episode. The production tables report finite gradients and losses on larger episodes, together with measured full-size time and peak GPU memory. Blank agreement values indicate that a paired calculation was unavailable.

In [4]:
from src.visualize import results2 as r2
agreement = c2.precision_agreement(summary)
fig = r2.precision_agreement(agreement, title='Mixed-precision gradients against FP32 at update zero')
cp.show(sink, c2.pages(fig, 'precision_agreement',
    'Cosine between mixed-precision and FP32 training gradients at the pretrained weights on identical episodes '
    'of up to 8,192 rows from partition 0. Each point represents one measured table, task, model and policy. '
    'Both FP16 policies use loss scaling. The FP32-normalization variant applies to TabPFN v2 PD. TabICLv2 '
    'keeps its column embedder in FP32. A cosine of 1 indicates identical directions. The dashed line marks '
    'the 0.90 agreement threshold. The table lists unavailable comparisons and identifies the training policy '
    'of the recorded run.'))
report.table('Mixed-precision comparisons on bounded episodes', agreement)
strict_production, native_parity = c2.precision_fp32_evidence(summary)
report.table('TabPFN v2 PD training loss and gradient under bounded strict FP32', strict_production)
report.table('Independent check of native and bounded FP32 on the same smaller episode', native_parity)
report.table('Full training episodes: recorded policy, gradient agreement, time and peak GPU memory',
             c2.precision_full_episodes(summary))
report.add('3. Training precision against FP32', c2.evidence_note('recorded precision probes', agreement)
           if not agreement.empty else c2.evidence_note('bounded strict FP32 training probes', strict_production))

No measurements available for this section yet.


""


""


""


""


## Summary

Every displayed table and figure, in section order.

In [5]:
print(report.summary(sink))

Experiment 0: second-campaign checks

1. Pilot audits and GPU checks
The local output has no Experiment 0 summary receipt.

2. LGD density roughness
The local summary contains no density measurements.

Table: Density roughness
CSV (first column: row index)
row

3. Training precision against FP32
No results yet for bounded strict FP32 training probes; the figure is drawn once they exist.

No measurements available for this figure group.

Table: Mixed-precision comparisons on bounded episodes
CSV (first column: row index)
row

Table: TabPFN v2 PD training loss and gradient under bounded strict FP32
CSV (first column: row index)
row

Table: Independent check of native and bounded FP32 on the same smaller episode
CSV (first column: row index)
row

Table: Full training episodes: recorded policy, gradient agreement, time and peak GPU memory
CSV (first column: row index)
row

experiment0/01_campaign2_checks: no figures saved.
